# 01 · Explore a dataset and train a baseline

Starter notebook for the project. It uses the bundled synthetic dataset
(`datasets/examples/synthetic_classification.csv`), so it runs without the database or the API.
Run it with the project's virtual environment as the kernel.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA = Path("..") / "datasets" / "examples" / "synthetic_classification.csv"
df = pd.read_csv(DATA)
print(df.shape)
df.head()

## Data quality

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
summary

In [ ]:
df.describe(include="all").T

## Target balance and feature distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.countplot(data=df, x="label", ax=axes[0])
axes[0].set_title("Target balance")
sns.histplot(data=df, x="num_a", hue="label", kde=True, ax=axes[1])
axes[1].set_title("num_a by label")
plt.tight_layout()
plt.show()

## Baseline model

Same idea as the API's training pipeline: impute + scale numeric columns, one-hot encode
categorical ones, fit inside a single scikit-learn `Pipeline` so there is no leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = "label"
X, y = df.drop(columns=[target]), df[target]
numeric = X.select_dtypes("number").columns.tolist()
categorical = X.columns.difference(numeric).tolist()

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore")), categorical),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

## Next steps

- Upload a dataset and train through the API (`POST /api/v1/datasets`, `POST /api/v1/models/train`)
  so the model is versioned, audited and servable.
- Local LLM: `POST /api/v1/llm/chat` uses Ollama (`llama3.2:3b`); no API key is needed.